# Producer Pattern 3 of 5 — Singleton / Shared Kafka Producer
### Zero → understanding → professional implementation | `confluent-kafka` + Python

> **Business scenario:** An order service has several functions (`create_order`, `capture_payment`, `reserve_inventory`) that need to publish events to Kafka. Should each function create its own `Producer`, or should they share one?

**Answer in one line:** **Create a long-lived producer for each application process and Kafka configuration; share it among the components that publish through that configuration.**

This notebook builds that idea **one small step at a time**, with a working application design and broker-free tests at the end.

**Prerequisite knowledge (from Notebooks 1–2):**
- `produce()` enqueues; background networking sends.
- `producer.poll()` processes delivery events/callbacks.
- `flush()` waits for outstanding messages before shutdown.
- A delivery callback tells you whether Kafka acknowledged the message according to your `acks` policy.

**Important precision:** A shared producer is **an ownership/lifecycle pattern**, not a new Kafka API. There is no `SingletonProducer` class in `confluent-kafka`.

## Learning roadmap

| Section | Objective | Takeaway |
|---|---|---|
| 0 | Set up a safe lab | Never delete real topics to learn a pattern |
| 1 | Define “shared producer” | One long-lived instance, many publishers |
| 2 | Visualize the expensive anti-pattern | Do not construct a producer for every request |
| 3 | Share by passing dependencies | Prefer explicit dependency injection |
| 4 | Explain why sharing helps | Connection reuse, batching, bounded resources |
| 5 | Understand threads | Sharing a producer across threads is supported |
| 6 | Decide who calls `poll()` | One clear owner for servicing callbacks |
| 7 | Design startup → publish → shutdown | Producer lifetime belongs to the application |
| 8 | Implement a professional shared publisher | Reuse, delivery tracking, backpressure, shutdown |
| 9 | Try the real implementation | Send multiple event types using one instance |
| 10 | Prove behavior without Kafka | Deterministic offline tests |
| 11 | Integrate into FastAPI | App lifespan, dependency injection |
| 12 | Know the boundaries | Processes, forks, configs, multi-cluster |
| 13 | Avoid subtle mistakes | Design review and troubleshooting |
| 14 | Final engineering cheat sheet | Remember only the essentials |
| 15 | Practice + next notebooks | Router and resilience patterns |

**Run order:** Execute cells from top to bottom. Live Kafka exercises are **skipped safely** when no broker is reachable. Offline tests run without Kafka.

---
# Section 0 — Set up the lab (safe by default)

**Objective:** Prepare the imports, connect to an existing broker, and create a *dedicated* topic **only if missing**.

**Why this section exists:** The producer pattern should not be confused with Docker administration. Unlike some earlier labs, we **never delete and recreate a topic** here.

**Requirements:** `pip install confluent-kafka`; Docker Kafka or another Kafka broker reachable from Python. If Kafka is not running, you can still study all explanations and run the offline tests.

In [ ]:
# Standard library: JSON payloads, timing, logs, threads, and type hints.
import json
import logging
import os
import threading
import time
import uuid
from concurrent.futures import ThreadPoolExecutor
from typing import Any, Callable

# Kafka library: producer sends; admin client sets up our lab topic.
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("shared-producer-tutorial")

BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
TOPIC = "nb03.shared.orders"
print("Kafka bootstrap:", BOOTSTRAP)

### 0.1 Probe the broker and ensure the topic exists

Why not use `recreate_topic()`? Deleting a topic in the middle of an application tutorial is unsafe and teaches the wrong habit. This helper is **lab-only setup**, not something each HTTP request should run.

In [ ]:
def ensure_lab_topic(topic_name: str, partitions: int = 3) -> bool:
    """Return True if Kafka is reachable; create topic only if needed."""
    admin = AdminClient({"bootstrap.servers": BOOTSTRAP})

    try:
        metadata = admin.list_topics(timeout=5)
        if topic_name in metadata.topics:
            print(f"Topic already exists: {topic_name}")
            return True

        # replication_factor=1 is fine for a SINGLE-broker local lab only.
        future = admin.create_topics([
            NewTopic(topic_name, num_partitions=partitions, replication_factor=1)
        ])[topic_name]
        future.result(timeout=10)
        print(f"Created lab topic: {topic_name}")
        return True

    except Exception as error:
        print(f"Kafka lab is unavailable: {error}")
        print("Live exercises will be skipped; offline tests remain available.")
        return False


LIVE_READY = ensure_lab_topic(TOPIC)

---
# Section 1 — What does “one shared producer” actually mean?

**Objective:** Explain the pattern to another junior developer without referring to Python implementation details.

### Analogy: One company mailroom, many departments

Your company has three departments:
- **Orders** creates a new-order event.
- **Payments** publishes a payment-authorized event.
- **Inventory** publishes a stock-reserved event.

**Bad office design:** Each department rents a new delivery truck *for every single parcel*. Trucks keep being created; there are many separate outgoing queues.

**Good office design:** All three departments use **one company mailroom**. The mailroom reuses its delivery infrastructure and can combine compatible parcels into efficient outbound work.

```text
                  ONE PYTHON PROCESS

   OrderService    PaymentService    InventoryService
         |              |                 |
         +--------------+-----------------+
                        |
                        v
              +---------------------+
              | ONE SHARED PRODUCER |
              | internal queue      |
              | batching + network  |
              +---------------------+
                        |
                        v
                    KAFKA BROKER
```

**This is not the Singleton design pattern requirement that *exactly one object must exist globally*.** It is a practical policy: **one producer per process per equivalent Kafka connection/configuration** is a strong default.

---
# Section 2 — The anti-pattern: creating a producer on every request

**Objective:** Understand *why* an application with many requests should not create a fresh producer for each publish operation.

First, examine the bad design. This cell is **illustrative only** — do not run it as a benchmark or production example.

In [ ]:
# BAD DESIGN — READ, DO NOT RUN AS AN APPLICATION PATTERN.
#
# def create_order(order):
#     producer = Producer({"bootstrap.servers": BOOTSTRAP})
#     producer.produce("orders", value=json.dumps(order).encode())
#     producer.flush(10)  # Forces each request to wait for delivery.
#
# The next request creates ANOTHER producer, another independent queue,
# another network client, and another round of startup / connection work.
#
# Constructing multiple producers is technically valid. The mistake is
# doing it unnecessarily for every request in a long-running service.
print("Anti-pattern reviewed (no new Producer created).")

### What is actually wasteful?

| Producer per request | Shared producer |
|---|---|
| Many separate producer instances | One long-lived instance in this process |
| Each instance manages its own network state | Reuses existing producer network state |
| Batches are split between many producers | Better opportunity to batch across requests |
| Hard to coordinate shutdown and callbacks | One clear lifecycle owner |
| Easy to call `flush()` for every request | Efficient asynchronous publishing |

**Technical correction:** Kafka producers maintain connections **as needed**; they do **not necessarily open a connection to every broker immediately**. The inefficiency comes from repeatedly constructing independent clients with separate queues, buffers, metadata, and connection state.

**Remember:** A second producer is not itself a bug or a guaranteed “connection leak.” The problematic design is uncontrolled creation and poorly managed lifetimes.

---
# Section 3 — The easiest shared pattern: pass the same producer

**Objective:** Start with plain Python functions before introducing classes, factories, or frameworks.

We create **one Kafka `Producer`**, then pass that same object to two business functions. This is called **dependency injection**: the function receives what it needs rather than constructing it itself.

In [ ]:
# Two realistic events from the same application.

def order_created(order_id: str) -> dict:
    return {
        "event_id": str(uuid.uuid4()),
        "event_type": "order.created",
        "order_id": order_id,
        "source": "order-service",
    }


def payment_authorized(order_id: str) -> dict:
    return {
        "event_id": str(uuid.uuid4()),
        "event_type": "payment.authorized",
        "order_id": order_id,
        "source": "payment-service",
    }


def publish_event(producer: Producer, event: dict) -> None:
    """Publish using the producer WE RECEIVED (not a new producer)."""
    producer.produce(
        topic=TOPIC,
        key=event["order_id"].encode("utf-8"),
        value=json.dumps(event).encode("utf-8"),
    )
    # Service available events without waiting for each message.
    producer.poll(0)


if LIVE_READY:
    shared_raw_producer = Producer({
        "bootstrap.servers": BOOTSTRAP,
        "client.id": "notebook-03-basic-shared",
    })

    publish_event(shared_raw_producer, order_created("ORD-101"))
    publish_event(shared_raw_producer, payment_authorized("ORD-101"))

    remaining = shared_raw_producer.flush(15)
    print("Same producer used for both functions:", id(shared_raw_producer))
    print("Still outstanding:", remaining)
else:
    print("Skipped live publish. The key idea is passing ONE producer object.")

### Section 3 recap

- Business functions don't know how to build the Kafka connection.
- The application **creates** the producer and **injects** it into functions.
- The producer is reused, while the message content changes.
- `flush()` happens at the end of a batch/lifecycle, **not after every event**.

**Limit of this small example:** No per-message delivery confirmation yet. Notebook 2 already taught delivery callbacks; we will incorporate them into the professional design.

---
# Section 4 — Why does a shared producer help with batching?

**Objective:** Understand the resource benefit without memorizing Kafka networking internals.

**Batching** means collecting compatible messages and transmitting them together to reduce per-message overhead. Batches are usually organized by topic-partition (messages to different partitions are not combined into one Kafka record batch).

Two message sources sharing one producer contribute work to the **same producer's queues**. That creates more opportunity for efficient batching than giving every request a separate producer.

```text
BAD: request 1 -> Producer A -> mostly empty batch
     request 2 -> Producer B -> mostly empty batch
     request 3 -> Producer C -> mostly empty batch

GOOD: request 1 --+
      request 2 --+--> Shared Producer --> eligible records batched
      request 3 --+    together by topic-partition
```

**Don't memorize throughput numbers.** Actual batching depends on `linger.ms`, `batch.size`, message rate, partitions, compression, and broker/network behavior. The correct engineering lesson is **reuse first, measure and tune later**.

---
# Section 5 — Can several Python threads share a producer?

**Objective:** Distinguish **threads** (share a process's memory) from **processes** (do not share the same normal Python objects).

Yes. `confluent_kafka.Producer` is **thread-safe**: different threads can call `produce()` on the same object.

One caveat: callbacks may execute in whichever application thread services producer events. Therefore:
- Keep callbacks short and non-blocking.
- Protect shared Python counters/state with locks if threads may update them.
- Decide who is responsible for calling `poll()`; don't scatter uncontrolled event servicing throughout the app.

Let's run a small real experiment using **one producer and multiple worker threads**.

In [ ]:
def publish_from_worker(shared: Producer, worker_number: int) -> str:
    """Every worker uses the SAME producer instance."""
    event = {
        "event_id": str(uuid.uuid4()),
        "worker": worker_number,
        "event_type": "shared.worker_event",
    }
    shared.produce(
        topic=TOPIC,
        key=f"worker-{worker_number}",
        value=json.dumps(event).encode("utf-8"),
    )
    return f"Worker {worker_number} used producer {id(shared)}"


if LIVE_READY:
    with ThreadPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(
            lambda worker: publish_from_worker(shared_raw_producer, worker),
            range(8),
        ))
    print(*results, sep="\n")
    print("Outstanding after flush:", shared_raw_producer.flush(15))
else:
    print("Skipped live thread test; see the offline thread-safety test later.")

**What it proves:** Multiple threads used the same **Python producer object**, and the client supports this pattern. It does **not** prove that one shared producer is always faster than multiple producers. Actual throughput depends on workload, partitions, and contention; both strategies are supported.

---
# Section 6 — Who should call `poll()` in a shared-producer design?

**Objective:** Avoid an easy-to-miss lifecycle problem.

With a short script, you can do this:

```python
producer.produce(..., on_delivery=callback)
producer.poll(0)
```

But in a **web service**, there may be long periods with no requests. If only request handlers call `poll()`, delivery callbacks might wait until another request arrives.

**Professional option:** Give the shared-producer component a **dedicated event-polling thread**. The background Kafka networking still does the actual sending. The extra application thread only calls `producer.poll()` to process delivery events regularly.

```text
                      SharedKafkaPublisher
                              |
                   +----------+-----------+
                   |                      |
            publish(...)              Polling thread
                   |                      |
          Producer.produce()         Producer.poll()
                   |                      |
          Kafka background          Your delivery callback
          networking                updates metrics/logs
```

**These are different threads/jobs:**
- `librdkafka` background threads: send messages, receive broker responses.
- Our *Python polling thread*: process delivery callbacks.

Another valid design is for the application to call `poll(0)` frequently itself. We choose a dedicated poller here because it gives a web application **one clear callback owner**.

---
# Section 7 — Application lifecycle: create once, reuse, shut down once

**Objective:** Define the ownership rule *before* writing the full class.

A reliable application lifecycle has only three stages:

1. **STARTUP:** Create the shared producer and start its event poller.
2. **RUNNING:** Inject the shared producer into many services; publish asynchronously.
3. **SHUTDOWN:** Reject new publishes, stop the polling thread, and `flush(timeout)` outstanding messages.

```text
Application starts
       |
       v
Create SharedKafkaPublisher   (once)
       |
       v
App services all reuse it    (many times)
       |
       v
Application shutdown signal
       |
       +--> Stop accepting new publishes
       +--> Stop callback poller
       +--> flush(remaining timeout)
       +--> inspect remaining / failures
```

**Why stop accepting new messages before flushing?** Otherwise a request could add another message while shutdown is trying to drain the queue.

**Why not `flush()` after each request?** It turns an efficient asynchronous producer into a request-by-request waiting pattern.

---
# Section 8 — Implement the professional shared producer

**Objective:** Build a small but realistic reusable component. **Read each responsibility first:**

| Component | Why we need it |
|---|---|
| `__init__` | Construct one `Producer` and start an event-polling thread |
| `publish()` | Serialize event, enqueue it, handle local queue pressure |
| `_on_delivery()` | Track broker-confirmed success/failure (Notebook 2) |
| `_poll_loop()` | Ensure callbacks are processed even with no incoming requests |
| `snapshot()` | Obtain thread-safe operational counters |
| `close()` | Stop new publishing and drain outstanding work |

**Deliberate scope:** This is a professional baseline, not a full infrastructure platform. It does not implement a durable outbox, a DLQ, circuit breaker, or distributed exactly-once delivery. Those topics belong to Notebook 5.

**Code design:** No unexplained decorators, no global magical singleton metaclass. Creation is explicit and testing is possible because we can inject a fake producer factory.

In [ ]:
class PublisherClosedError(RuntimeError):
    """Raised when publishing is attempted after shutdown began."""


class SharedKafkaPublisher:
    """One long-lived Kafka producer with bounded enqueue and a poller.

    Each app process creates this class ONCE per Kafka configuration.
    Services receive the same instance through dependency injection.
    """

    def __init__(
        self,
        config: dict,
        *,
        producer_factory: Callable = Producer,
        poll_interval_seconds: float = 0.2,
        log: logging.Logger = logger,
    ):
        if poll_interval_seconds <= 0:
            raise ValueError("poll_interval_seconds must be positive")

        # The factory lets tests inject a fake instead of contacting Kafka.
        self._producer = producer_factory(config)
        self._log = log
        self._poll_interval = poll_interval_seconds

        # Protect shutdown/publish coordination.
        self._state_lock = threading.Lock()
        self._close_lock = threading.Lock()
        self._closing = False

        # Delivery callbacks run on the polling thread (or during flush).
        # The lock keeps counter updates consistent across threads.
        self._metrics_lock = threading.Lock()
        self._metrics = {
            "accepted": 0,           # produce() accepted locally
            "delivered": 0,          # confirmed by delivery callback
            "delivery_failed": 0,    # callback reported final failure
            "enqueue_rejected": 0,   # local queue remained full too long
        }

        self._stop_poller = threading.Event()
        self._poller = threading.Thread(
            target=self._poll_loop,
            name="kafka-producer-event-poller",
            daemon=True,
        )
        self._poller.start()

    def _poll_loop(self) -> None:
        """Service callbacks even while the app receives no requests."""
        while not self._stop_poller.is_set():
            try:
                self._producer.poll(self._poll_interval)
            except Exception:
                # Keep this event thread alive, but make failures visible.
                self._log.exception("Unexpected error while polling producer")
                self._stop_poller.wait(self._poll_interval)

    def _on_delivery(self, event_id: str, error, message) -> None:
        """Delivery reports do not run until poll()/flush() services them."""
        with self._metrics_lock:
            if error is None:
                self._metrics["delivered"] += 1
            else:
                self._metrics["delivery_failed"] += 1

        if error is not None:
            # Production: log failure + alert/metrics. A durable retry policy
            # requires additional design (Notebook 5).
            self._log.error("Delivery failed for %s: %s", event_id, error)

    def publish(
        self,
        *,
        topic: str,
        key: str,
        event: dict,
        enqueue_timeout_seconds: float = 3.0,
    ) -> None:
        """Accept an event asynchronously, or raise a clear error.

        Returning successfully means *locally accepted*, NOT broker-delivered.
        """
        if enqueue_timeout_seconds < 0:
            raise ValueError("enqueue_timeout_seconds cannot be negative")
        if not topic or not key:
            raise ValueError("topic and key must be non-empty")
        if not isinstance(event, dict):
            raise TypeError("event must be a dictionary")

        payload = json.dumps(event, separators=(",", ":")).encode("utf-8")
        key_bytes = key.encode("utf-8")
        event_id = str(event.get("event_id", "unknown"))
        deadline = time.monotonic() + enqueue_timeout_seconds

        while True:
            # Prevent a request from publishing after shutdown has begun.
            with self._state_lock:
                if self._closing:
                    raise PublisherClosedError("Publisher is shutting down")

                try:
                    # Use a counter lock so a very fast callback cannot
                    # be counted before local acceptance is counted.
                    with self._metrics_lock:
                        self._producer.produce(
                            topic=topic,
                            key=key_bytes,
                            value=payload,
                            on_delivery=lambda error, message: self._on_delivery(
                                event_id, error, message
                            ),
                        )
                        self._metrics["accepted"] += 1
                    return

                except BufferError:
                    # Local queue is temporarily saturated. We retry below
                    # WITHOUT holding locks so the poller can make progress.
                    pass

            if time.monotonic() >= deadline:
                with self._metrics_lock:
                    self._metrics["enqueue_rejected"] += 1
                raise TimeoutError(
                    f"Kafka local queue stayed full for {enqueue_timeout_seconds}s"
                )

            # The poller handles delivery callbacks; we briefly wait for
            # network progress instead of spinning at 100% CPU.
            time.sleep(min(0.05, max(0.0, deadline - time.monotonic())))

    def snapshot(self) -> dict:
        """Return a thread-safe COPY of the delivery counters."""
        with self._metrics_lock:
            return dict(self._metrics)

    def close(self, timeout_seconds: float = 20.0) -> int:
        """Stop new publishes and drain pending work.

        Returns count of outstanding messages/events after flush().
        If timeout occurs, a later close() MAY be retried; publishes stay off.
        """
        if timeout_seconds < 0:
            raise ValueError("timeout_seconds cannot be negative")

        # Only one thread may execute close() at a time.
        with self._close_lock:
            with self._state_lock:
                self._closing = True

            self._stop_poller.set()
            self._poller.join(timeout=max(1.0, self._poll_interval * 3))
            if self._poller.is_alive():
                raise RuntimeError("Producer polling thread did not stop")

            # flush() itself executes any pending delivery callbacks.
            remaining = self._producer.flush(timeout_seconds)
            if remaining:
                self._log.warning("Shutdown left %s outstanding messages/events", remaining)
            return remaining

### Read the implementation in five passes

1. **Only one Kafka client:** `producer_factory(config)` runs once in `__init__`; `publish()` never constructs another producer.
2. **No per-message blocking:** `publish()` calls `produce()` and returns on local acceptance. It does not call `flush()`.
3. **One callback owner:** `_poll_loop()` continuously services events, so callbacks are handled even without web requests.
4. **Backpressure is bounded:** A full local queue retries briefly; if it stays full, `publish()` raises `TimeoutError` rather than hanging forever.
5. **Shutdown is coordinated:** New publishes are rejected before `flush()` starts. A nonzero return means delivery work remains; zero does **not** mean every delivery succeeded, so check `snapshot()`.

**What the class does not promise:** The in-memory queue is not durable if the process crashes. `enable.idempotence` does not make your business operation end-to-end exactly once; it protects eligible producer retries. Delivery failures must have an explicit business recovery policy.

**Why locks?** We allow multiple request threads to call `publish()`. The Kafka `Producer` itself is thread-safe, but **our application state** (`_closing` and counters) also needs coordination.

---
# Section 9 — Real usage: three services, one producer instance

**Objective:** Apply the component to several event types without rebuilding the Kafka client.

The application creates the shared publisher **one time**, then gives it to business functions.

In [ ]:
def app_create_order(publisher: SharedKafkaPublisher, order_id: str) -> None:
    """Simulate the HTTP order endpoint / business function."""
    publisher.publish(topic=TOPIC, key=order_id, event={
        "event_id": str(uuid.uuid4()),
        "event_type": "order.created",
        "order_id": order_id,
    })


def app_capture_payment(publisher: SharedKafkaPublisher, order_id: str) -> None:
    """Another business function reuses the SAME publisher."""
    publisher.publish(topic=TOPIC, key=order_id, event={
        "event_id": str(uuid.uuid4()),
        "event_type": "payment.captured",
        "order_id": order_id,
    })


def app_reserve_inventory(publisher: SharedKafkaPublisher, order_id: str) -> None:
    """A third function also reuses the SAME publisher."""
    publisher.publish(topic=TOPIC, key=order_id, event={
        "event_id": str(uuid.uuid4()),
        "event_type": "inventory.reserved",
        "order_id": order_id,
    })


if LIVE_READY:
    app_publisher = SharedKafkaPublisher({
        "bootstrap.servers": BOOTSTRAP,
        "client.id": "nb03-order-app",
        "acks": "all",
        "enable.idempotence": True,
        "delivery.timeout.ms": 30000,
    })

    try:
        for number in range(1, 6):
            order_id = f"ORD-{number:04d}"
            app_create_order(app_publisher, order_id)
            app_capture_payment(app_publisher, order_id)
            app_reserve_inventory(app_publisher, order_id)

        # Publishing is asynchronous. The snapshot may show deliveries
        # arriving gradually while our poller processes acknowledgements.
        print("During work:", app_publisher.snapshot())

    finally:
        outstanding = app_publisher.close(timeout_seconds=35)
        print("After shutdown:", app_publisher.snapshot())
        print("Outstanding messages/events:", outstanding)
else:
    print("Skipped live application. Continue to the offline tests.")

### What did we prove?

- Three business functions reused one `SharedKafkaPublisher`.
- The publisher's single underlying `Producer` was constructed **once**.
- The main application did not call `poll()`; the component's poller handled callback servicing.
- At shutdown, we stopped accepting work before flushing.

**Read the statistics carefully:** `accepted` means *locally queued*. `delivered` means a successful delivery callback was processed. `delivery_failed` is a final delivery error. Only a completed drain and `accepted == delivered + delivery_failed` means every accepted event reached a final reported outcome; **some of those outcomes can still be failures**.

---
# Section 10 — Offline tests (NO running broker required)

**Objective:** Verify the *design properties* instead of hoping a real broker behaves a particular way.

We will replace the Kafka client with a **FakeProducer** that has the same three operations we use: `produce()`, `poll()`, and `flush()`.

This is called **dependency injection for testability**. The fake does not prove Kafka network integration; it proves *our own application code* manages reuse, callbacks, backpressure, and shutdown correctly.

In [ ]:
class FakeProducer:
    """Tiny test double: stores messages and returns scripted delivery results."""

    def __init__(self, config, *, fail_keys=(), always_full=False):
        self.config = config
        self.fail_keys = set(fail_keys)
        self.always_full = always_full
        self.messages = []
        self.lock = threading.Lock()
        self.creation_count = 1

    def produce(self, *, topic, key, value, on_delivery):
        if self.always_full:
            raise BufferError("Simulated local queue full")
        with self.lock:
            self.messages.append((topic, key, value, on_delivery))

    def poll(self, timeout):
        with self.lock:
            item = self.messages.pop(0) if self.messages else None

        if item is None:
            time.sleep(min(max(timeout, 0), 0.01))
            return 0

        topic, key, value, callback = item
        error = "Simulated broker rejection" if key in self.fail_keys else None
        callback(error, object())
        return 1

    def flush(self, timeout):
        deadline = time.monotonic() + timeout
        while True:
            with self.lock:
                remaining = len(self.messages)
            if remaining == 0 or time.monotonic() >= deadline:
                return remaining
            self.poll(0)

### Test A — One instance reused across multiple service calls

We check that many calls result in **one producer construction**, and that accepted events produce delivery results.

In [ ]:
created_clients = []


def fake_factory(config):
    fake = FakeProducer(config)
    created_clients.append(fake)
    return fake


publisher = SharedKafkaPublisher(
    {"bootstrap.servers": "offline"},
    producer_factory=fake_factory,
    poll_interval_seconds=0.01,
)

for order_id in range(10):
    app_create_order(publisher, f"TEST-{order_id}")
    app_capture_payment(publisher, f"TEST-{order_id}")

remaining = publisher.close(timeout_seconds=1)
stats = publisher.snapshot()

assert len(created_clients) == 1, "Expected one shared producer"
assert stats["accepted"] == 20
assert stats["delivered"] == 20
assert stats["delivery_failed"] == 0
assert remaining == 0

print("PASS: 20 events, one underlying producer, 20 completed deliveries")

### Test B — Failure callbacks are counted, not hidden

This fake intentionally rejects a specific key so we can verify that the application notices delivery failure **even when flush returns zero**.

In [ ]:
fake_with_failure = FakeProducer({}, fail_keys={b"ORD-FAIL"})

publisher = SharedKafkaPublisher(
    {},
    producer_factory=lambda config: fake_with_failure,
    poll_interval_seconds=0.01,
)

publisher.publish(
    topic=TOPIC,
    key="ORD-OK",
    event={"event_id": "evt-ok", "event_type": "order.created"},
)
publisher.publish(
    topic=TOPIC,
    key="ORD-FAIL",
    event={"event_id": "evt-fail", "event_type": "order.created"},
)

remaining = publisher.close(timeout_seconds=1)
stats = publisher.snapshot()

assert remaining == 0
assert stats["accepted"] == 2
assert stats["delivered"] == 1
assert stats["delivery_failed"] == 1

print("PASS: flush=0 but one delivery failed (correctly tracked)")

### Test C — Backpressure cannot freeze the application forever

If `produce()` continuously raises `BufferError`, our wrapper has a **deadline** rather than an infinite retry loop.

In [ ]:
always_full_client = FakeProducer({}, always_full=True)
publisher = SharedKafkaPublisher(
    {},
    producer_factory=lambda config: always_full_client,
    poll_interval_seconds=0.01,
)

try:
    try:
        publisher.publish(
            topic=TOPIC,
            key="ORD-QUEUE-FULL",
            event={"event_id": "evt-full"},
            enqueue_timeout_seconds=0.1,
        )
        raise AssertionError("Expected TimeoutError")
    except TimeoutError:
        pass  # Expected outcome: queue pressure was not recoverable in time.
finally:
    publisher.close(timeout_seconds=1)

assert publisher.snapshot()["enqueue_rejected"] == 1
print("PASS: persistent local queue pressure fails within a deadline")

### Test D — Parallel requests share one producer safely

This test checks our wrapper's counters and lifecycle under threads. The fake itself is protected by a lock; Kafka's real producer is thread-safe too.

In [ ]:
client = FakeProducer({})
publisher = SharedKafkaPublisher(
    {},
    producer_factory=lambda config: client,
    poll_interval_seconds=0.01,
)

with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(
        lambda i: app_create_order(publisher, f"PAR-{i}"),
        range(80),
    ))

remaining = publisher.close(timeout_seconds=2)
counts = publisher.snapshot()

assert remaining == 0
assert counts["accepted"] == 80
assert counts["delivered"] == 80
assert counts["delivery_failed"] == 0

print("PASS: 8 request threads, 80 accepted events, 80 callbacks")

### Test E — Shutdown rejects new messages

Once `close()` begins, no request should slip in and queue another event. If a shutdown times out, subsequent `close()` calls may attempt another drain; publishing still remains disabled.

In [ ]:
client = FakeProducer({})
publisher = SharedKafkaPublisher(
    {},
    producer_factory=lambda config: client,
    poll_interval_seconds=0.01,
)

assert publisher.close(timeout_seconds=1) == 0

try:
    publisher.publish(
        topic=TOPIC,
        key="TOO-LATE",
        event={"event_id": "evt-late"},
    )
    raise AssertionError("Expected PublisherClosedError")
except PublisherClosedError:
    pass

assert publisher.close(timeout_seconds=1) == 0  # repeat close is supported
print("PASS: publisher prevents sends after shutdown")

### What these tests do and do not guarantee

**They verify:** one client construction, callback counters, bounded queue pressure, multi-thread publishing, and shutdown gating.

**They don't verify:** real TCP connection reuse, broker availability, partition ordering, Kafka-level idempotence, actual batching efficiency, or infrastructure failures. Those need integration/load tests with a real Kafka cluster.

---
# Section 11 — Production application integration (FastAPI lifespan)

**Objective:** Know **where** to construct the shared producer in a real web application.

FastAPI provides an application **lifespan**: one startup and one shutdown for an application worker process. We create a publisher in startup, store it on `app.state`, and inject it into request handlers.

**This is reference implementation, not a notebook cell to run directly.** Put it in `app.py` inside an application where `fastapi` is installed.

```python
from contextlib import asynccontextmanager
import os
import uuid
from fastapi import FastAPI, Depends, Request
from pydantic import BaseModel

# In a project, import SharedKafkaPublisher from your infrastructure module.

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Once per FastAPI worker PROCESS, not once per HTTP request.
    publisher = SharedKafkaPublisher({
        "bootstrap.servers": os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092"),
        "acks": "all",
        "enable.idempotence": True,
        "delivery.timeout.ms": 30000,
    })
    app.state.publisher = publisher

    try:
        yield  # FastAPI now serves requests.
    finally:
        # flush() is blocking; run it outside the async event loop.
        import asyncio
        remaining = await asyncio.to_thread(publisher.close, 35)
        if remaining:
            # Emit an alert / metric and define an operational response.
            print(f"WARNING: {remaining} Kafka events remain at shutdown")

app = FastAPI(lifespan=lifespan)

class OrderInput(BaseModel):
    order_id: str


def get_publisher(request: Request) -> SharedKafkaPublisher:
    return request.app.state.publisher

@app.post("/orders", status_code=202)
def create_order_endpoint(
    order: OrderInput,
    publisher: SharedKafkaPublisher = Depends(get_publisher),
):
    publisher.publish(
        topic="orders.events",
        key=order.order_id,
        event={
            "event_id": str(uuid.uuid4()),
            "event_type": "order.created",
            "order_id": order.order_id,
        },
    )

    # HTTP 202 means accepted for asynchronous processing;
    # it does NOT claim the broker has acknowledged the event.
    return {"status": "queued", "order_id": order.order_id}
```

### Why this is better than a `@singleton` decorator everywhere

- **Ownership is explicit:** the application creates and closes the publisher.
- **Tests are easy:** replace the dependency with a fake.
- **Request handlers stay focused** on business logic.
- **No accidental creation on import** (particularly important during tests and process startup).

**Real-world consideration:** If Kafka event publication must be atomic with a database transaction, an application-wide shared producer alone does not solve it. Use an outbox or equivalent consistency design.

---
# Section 12 — The boundaries of “one producer”

**Objective:** Prevent the “singleton” idea from turning into an incorrect absolute rule.

| Situation | Appropriate pattern | Why |
|---|---|---|
| One service process, one Kafka cluster/config | **One long-lived producer** (good default) | Reuse state, queues, batching |
| Several service modules in same process | **Inject the same producer** | Reuse without global state |
| Several Python threads in one process | **Sharing is supported** | `Producer` is thread-safe |
| Four Gunicorn/Uvicorn worker processes | **One producer in each worker** | Each worker is a separate process |
| Separate Kafka clusters / credentials | **Distinct producer per configuration** | Different connections/security contexts |
| Distinct transactional identities | **Separate producers as needed** | Transactional ownership/config differs |
| High throughput requiring careful tuning | **Benchmark the design** | Multiple producers may be justified |

### Processes are not threads

```text
                       MACHINE
                         |
          +--------------+--------------+
          |              |              |
      Process A      Process B      Process C
          |              |              |
      Producer A     Producer B     Producer C
```

**Do not construct a producer in a parent process and try to reuse that active client after a process fork.** Create it inside the worker after startup/fork. Kafka clients include native threading/network state and are not ordinary picklable shared objects.

**What to say in an interview:** “I usually keep one Kafka producer per service process and configuration, sharing it through dependency injection. I create it at startup, handle callbacks centrally, and flush during graceful shutdown.”

---
# Section 13 — Production design review: mistakes to avoid

| Mistake | Why it's harmful | Better approach |
|---|---|---|
| New `Producer()` for every request | Separate resource pools and missed batching opportunities | Shared long-lived producer |
| `flush()` after every message | Turns asynchronous throughput into synchronous waiting | `produce()` normally; `flush()` at shutdown |
| No callback servicing | Delivery reports may stay unprocessed | Regular `poll()` or one event-polling thread |
| Callbacks perform database/network I/O | Blocks callback servicing | Keep callbacks short; queue follow-up work |
| Ignore `BufferError` | Surprising failures under load | Bounded backpressure policy |
| `flush() == 0` interpreted as all succeeded | Finished does not equal successful | Track callback failure count |
| One global producer imported during module load | Hard testing, uncertain startup/fork behavior | Create on app startup and inject |
| Same client assumed across processes | Processes do not share ordinary Python memory | Create inside each process |
| No explicit shutdown path | Pending messages may be lost when process exits | Coordinate stop-and-drain |
| Confuse local queue success with durable event | A crash can lose queued messages | Outbox for business-critical persistence |

### Operational metrics worth collecting

- **Accepted** (successfully enqueued locally)
- **Delivered** (successful callback)
- **Failed** (delivery callback with error)
- **Enqueue rejected** (queue-full deadline exceeded)
- **Outstanding at shutdown** (`flush()` returned nonzero)
- **Producer queue depth, latency, error rates** (as your observability needs grow)

### Practical deployment checklist

- [ ] Producer is created during application startup, not in each request.
- [ ] It is reused by application components that share Kafka configuration.
- [ ] Callbacks are serviced even when the request rate falls to zero.
- [ ] Callbacks update metrics/logs and never block for long.
- [ ] There is a bounded response to full local queues.
- [ ] Startup/shutdown ownership is defined.
- [ ] Shutdown rejects new work, drains, and checks the result.
- [ ] The recovery policy for failed delivery is documented.

---
# Section 14 — What engineers actually remember (one-page cheat sheet)

## The five concepts worth retaining

1. **Reuse a producer.** One long-lived `Producer` per process and equivalent Kafka configuration is a good default.
2. **Share by injection.** Build the client at application startup, pass the same instance to many services/functions.
3. **Preserve asynchronous publishing.** `produce()` enqueues; don't `flush()` for every request.
4. **Own delivery reports.** Central polling or a documented polling strategy processes callbacks; use those callbacks for error visibility.
5. **Shut down safely.** Stop new messages, drain with bounded `flush()`, inspect failures and remaining work.

### Three anti-patterns worth recognizing

- `Producer()` inside every HTTP request handler.
- `producer.flush()` immediately after every `produce()` in a high-throughput service.
- A shared producer with no owner responsible for polling and shutdown.

### Architecture mnemonic: **ONE → SHARE → POLL → DRAIN**

- **ONE:** Create a long-lived producer per process/configuration.
- **SHARE:** Inject it into modules and request handlers.
- **POLL:** Process delivery reports without blocking request handling.
- **DRAIN:** On shutdown, stop publishing and `flush(timeout)`.

> **Interview-ready answer:** “A shared Kafka producer centralizes buffering, connection management, batching, delivery reporting, and shutdown for a service process. I create it during startup, inject it into components, process callbacks continuously, and perform a bounded drain on shutdown.”

---
# Section 15 — Practice questions and the next two notebooks

**Explain these without looking at the answer:**

1. If ten HTTP requests arrive together, should you create ten `Producer` objects?
2. Does using one producer mean only one message can be sent at a time?
3. Is sharing a producer across Python threads safe?
4. Does one shared producer cover four worker processes?
5. Why is `flush()` inside each request handler an anti-pattern?
6. Who executes delivery callbacks if HTTP traffic stops?
7. Does a single shared producer guarantee Kafka delivery or exactly-once business processing?
8. When would more than one producer in a process be reasonable?

<details>
<summary>Check your answers</summary>

1. No; reuse the application's producer for the same Kafka configuration.
2. No; the producer queues and batches messages asynchronously.
3. Yes, `confluent_kafka.Producer` supports sharing across threads.
4. No; each process should construct its own producer.
5. It waits for outstanding delivery and sacrifices async throughput.
6. A dedicated polling worker, or whichever application thread explicitly calls `poll()`/`flush()`.
7. No; callbacks, error recovery, durability design and consumer semantics matter.
8. Different clusters/configurations/transactional IDs or an evidenced throughput need.

</details>

## Where this pattern fits in the five-notebook series

| Notebook | Pattern | Question answered |
|---|---|---|
| **01** | Fire-and-Forget | How do I publish asynchronously? |
| **02** | Callback-Confirmed | How do I know whether Kafka accepted it? |
| **03 (this one)** | Singleton / Shared Producer | How do several modules share Kafka infrastructure correctly? |
| **04 (next)** | Content-Based Router | How do I choose the correct Kafka topic from event content? |
| **05 (after that)** | Resilient Producer / DLQ | What do I do when delivery keeps failing or the broker is unavailable? |

**Design connection:** Notebook 4 can be built as a routing component that receives the `SharedKafkaPublisher` from this notebook. It chooses a topic and calls the same `publish()` method. Notebook 5 builds on that foundation with durable error-handling and recovery policy.

---
## References (official documentation)

1. [Confluent Kafka Python API — Producer `produce()`, `poll()`, `flush()`](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html)
2. [Confluent Python Client — asynchronous writes and callbacks](https://docs.confluent.io/kafka-clients/python/current/overview.html)
3. [Confluent Python Client — multithreading guide](https://github.com/confluentinc/confluent-kafka-python/blob/master/docs/multithreading-guide.md)

> **Scope and guarantees:** These examples provide a testable shared-producer baseline. They are not a full guarantee of durability, ordering across partitions, transactional atomicity, or exactly-once application semantics. Tune configuration, resilience, logging, and monitoring for your deployment.